### Minicurso Sistemas Multi-agente com LangGraph

### Parte 2

## Agente REACT (Reasoning + Acting)

Moacir Antonelli Ponti - 2025

---

No LangGraph, um agente ReAct é qualquer nó que:
- está ligado a ferramentas
- e cujo fluxo faz: LLM > Tool > LLM > ... até finalizar.

In [ ]:
import os
from typing import Annotated, Sequence, TypedDict
from langchain_core.messages import BaseMessage # base class for messages in LangChain
from langchain_core.messages import ToolMessage # passes data back to LLM afer it calls a tool 
from langchain_core.messages import SystemMessage # message providing instruction to the LLM
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from IPython.display import Image, display
from dotenv import load_dotenv

load_dotenv()

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [ ]:
# decorador definindo a funcao como ferramenta
# docstring sera usado como descricao da ferramenta
# - sao fundamentais para o LLM entender quando usar a ferramenta
@tool
def add(a: float, b: float):
    """Addition function that adds 2 number together"""
    print(f"] tool: Adding {a} + {b}")
    return a + b

# divide tool
# create list
# create LLM + tool binding

In [ ]:
# model call node

# decisor node (should continue)

In [ ]:
graph = StateGraph(AgentState)
graph.add_node("agent", model_call)

# Tool Node

agent = graph.compile()

In [ ]:
display(Image(agent.get_graph().draw_mermaid_png()))

In [ ]:
inputs = {"messages" : [("user", "add 5 and -2 and then sum 10 on this result")]}
res = agent.stream(inputs, stream_mode="values")

In [ ]:
for s in res:
    print(s["messages"][-1].content)

In [ ]:
inputs = {"messages" : [("user", "Calcule x = 300 + 100; então divida recursivamente o novo resultado por 2, ate obter um valor menor que 10")]}
res = agent.stream(inputs, stream_mode="values")

In [72]:
for s in res:
    print(s["messages"][-1].content)

## Exercício:

Adicione uma tool que converte USD para BRL e que permita fazer operações envolvendo câmbio, ex.:

In [ ]:

@tool
def cambio_usd_brl() -> str:
    """Cotação USD BRL via API pública."""
    url = "https://economia.awesomeapi.com.br/json/last/USD-BRL"
    r = requests.get(url).json()
    valor = float(r["USDBRL"]["bid"])
    return f"1 USD = {valor:.2f} BRL"


In [ ]:
inputs = {"messages" : [("user", "Converta 100 dólares em reais, e some 45 reais ao resultado, divida o total por 2")]}
res = agent.stream(inputs, stream_mode="values")